# 🚀 Train Your First Custom Wake Word with Nanowakeword!

Welcome to the official tutorial for **Nanowakeword**!

In this notebook, we will guide you through the entire process of training a high-performance, custom wake word model from scratch. You don't need any pre-existing data—we will download everything we need and let Nanowakeword's intelligent engine do the heavy lifting.

**Our goal:** Go from zero to a ready-to-use wake word model in just a few simple steps. Let's get started!

**Installation**

In [1]:
# @title Step 1: Install Nanowakeword
# We install the full [train] package to get all the necessary dependencies.

! pip install "nanowakeword[train] @ git+https://github.com/arcosoph/nanowakeword.git"
! pip install piper-tts
# ! huggingface_hub, datasets

print("Installation complete!")

  Cloning https://github.com/arcosoph/nanowakeword.git to /tmp/pip-install-ikbhfhvx/nanowakeword_d2e82c8ff92349babef2a96c7b9265ed
  Running command git clone --filter=blob:none --quiet https://github.com/arcosoph/nanowakeword.git /tmp/pip-install-ikbhfhvx/nanowakeword_d2e82c8ff92349babef2a96c7b9265ed
  Resolved https://github.com/arcosoph/nanowakeword.git to commit 0637941b3b74143ea95e9cc47f191039b4250108
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 47.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Step 2: Prepare the Dataset

A great model starts with great data. For this tutorial, we will:
1.  **Download** open-source noise and Room Impulse Response (RIR) datasets.
2.  **Organize** all your project files within a clean, well-structured folder hierarchy for better clarity and maintainability.

In [ ]:
import shutil
import os, requests
from tqdm import tqdm
from datasets import load_dataset, Audio
from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor

# RIR data
# Disable HF symlink warning
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# Output directory
output_dir = "data/rir"
os.makedirs(output_dir, exist_ok=True)

# Load dataset (streaming, no decode)
dataset = load_dataset(
    "davidscripka/MIT_environmental_impulse_responses",
    split="train",
    streaming=True
)
dataset = dataset.cast_column("audio", Audio(decode=False))

repo_id = "davidscripka/MIT_environmental_impulse_responses"

for row in tqdm(dataset):
    hf_path = row["audio"]["path"]
    filename = os.path.basename(hf_path)
    save_path = os.path.join(output_dir, filename)

    if os.path.exists(save_path):
        continue

    # Download from HF cache (or repo)
    local_file = hf_hub_download(
        repo_id=repo_id,
        filename=hf_path.split("/")[-2] + "/" + filename,  # keep subfolder
        repo_type="dataset"
    )

    # Cross-drive safe copy
    shutil.copy(local_file, save_path)


# # noice data
# out_dir = "data/background_noise"
# os.makedirs(out_dir, exist_ok=True)

# # 1) Get file list from GitHub API
# api = "https://api.github.com/repos/karolpiczak/ESC-50/contents/audio"
# files = [f["name"] for f in requests.get(api).json() if f["name"].endswith(".wav")]

# base = "https://raw.githubusercontent.com/karolpiczak/ESC-50/master/audio/"

# def dl(fname):
#     path = os.path.join(out_dir, fname)
#     if os.path.exists(path): return
#     r = requests.get(base + fname, stream=True)
#     if r.status_code == 200:
#         with open(path, "wb") as f:
#             for c in r.iter_content(8192):
#                 f.write(c)

# # 2) Parallel download (SUPERFAST)
# with ThreadPoolExecutor(max_workers=16) as ex:
#     list(tqdm(ex.map(dl, files), total=len(files)))


In [ ]:
!wget https://huggingface.co/datasets/arcosoph/AE29H_float32/resolve/main/AE29H_float32.npy
!wget https://huggingface.co/datasets/arcosoph/RACON_11h_v1/resolve/main/RACON_11h_v1.npy

## Step 3: Configure and Train the Model

Now for the fun part! We will create a `config.yaml` file and then run the Nanowakeword training command.

In [ ]:
# @title Step 3.1: Create the "NanoWakeWord" Configuration File

# ⚠️ FULL config Explanation: https://arcosoph.com/blog/nanowakeword_config_guide#nanowakeword-configuration-guide

import yaml

config_dict = {
    # ==========================================================
    # Project & Data Paths
    # ==========================================================
    "model_name": "arcosoph_A_v1",
    "output_dir": "./trained_models",

    "positive_data_path": "./T_data/positive",
    "negative_data_path": "./T_data/negative",
    "background_paths": ["./final_clips"],
    "rir_paths": ["notebooks/data/rir"],

    # ==========================================================
    # Model Architecture
    # ==========================================================
    "model_type": "crnn",
    "layer_size": 64,
    "n_blocks": 4,
    "embedding_dim": 128,
    "dropout_prob": 0.2,
    "activation_function": "relu",

    # ==========================================================
    # Loss Function
    # ==========================================================
    "margin_pos": 2.5,
    "margin_neg": -2.5,
    "LOSS_BIAS": 0.70,

    "logit_reg_weight": 0.0005,
    "logit_reg_margin": 4.0,
    "logit_min_margin": 1.5,

    # ==========================================================
    # Training Settings
    # ==========================================================
    "steps": 60000,
    "stabilization_steps": 25000,

    "optimizer_type": "adamw",
    "learning_rate_max": 0.0008,
    "lr_scheduler_type": "onecycle",
    "weight_decay": 0.01,
    "momentum": 0.9,
    "num_workers": 0,

    # ==========================================================
    # Synthetic Data Generation
    # ==========================================================
    "target_phrase": "jaag_ruut",

    "data_generation_tasks": [
        {
            "name": "any name like pos_data",
            "enabled": True,
            "output_dir": "data/positive",
            "num_samples": 2500,
            "file_prefix": "pos",
            "text_source": {
                "type": "fixed_phrase",
                "phrase": "jaag_ruut"
            }
        },
        {
            "name": "pos_val",
            "enabled": True,
            "output_dir": "data/positive_val",
            "num_samples": 2000,
            "file_prefix": "pos",
            "text_source": {
                "type": "fixed_phrase",
                "phrase": "jaag_ruut"
            }
        },
        {
            "name": "Adversarial Negatives",
            "enabled": True,
            "output_dir": "data/negative",
            "num_samples": 5000,
            "file_prefix": "neg_auto",
            "text_source": {
                "type": "auto_adversarial",
                "base_phrase": "jaag_ruut",
                "include_input_words": True,
                "include_partial_phrase": True,
                "multi_word_prob": 0.5,
                "max_multi_word_len": 3
            }
        },
        {
            "name": "Phoneme Hard Negatives",
            "enabled": True,
            "output_dir": "data/negative_phoneme",
            "num_samples": 3000,
            "file_prefix": "neg_ph",
            "text_source": {
                "type": "phoneme_adversarial",
                "base_phrase": "jaag_ruut",
                "min_distance": 0.3
            }
        },
        {
            "name": "Custom Negatives",
            "enabled": True,
            "output_dir": "data/negative",
            "num_samples": 50,
            "file_prefix": "neg_custom",
            "text_source": {
                "type": "from_list",
                "phrases": [
                    "jaag_ruut",
                    "arcosop",
                    "hi arcosoph",
                    "arkhoshap",
                    "yarkasop"
                ],
                "repeat_each": 10
            }
        }
    ],

    # ==========================================================
    # Feature Manifest
    # ==========================================================
    "feature_manifest": {
        "targets": {
            "t": "./trained_models/arcosoph_A_v1/features/positive_features.npy"
        },
        "negatives": {
            "AE28H_float32": "./AE29H_float32.npy",
            "b": "./RACON_11h_v1.npy",
            "n": "./trained_models/arcosoph_A_v1/features/negative_features.npy",
            "hn": "./trained_models/arcosoph_A_v1/features/hard_negative_features.npy"
        },
        "targets_val": {
            "t_v": "./trained_models/arcosoph_A_v1/features/positive_features_val.npy"
        },
        "negatives_val": {
            "bv": "RACON_11h_v1.npy"
        }
    },

    # ==========================================================
    # Batch Composition
    # ==========================================================
    "batch_composition": {
        "AE28H_float32": 90,
        "b": 64,
        "hn": 48,
        "n": 80,
        "t": 64
    },

    # ==========================================================
    # Augmentation
    # ==========================================================
    "augmentation_batch_size": 16,
    "feature_gen_cpu_ratio": 1.0,

    "augmentation_settings": {
        "gain_prob": 1.0,
        "max_gain_in_db": 2.0,
        "max_pitch_semitones": 1.0,
        "max_snr_in_db": 35.0,
        "min_gain_in_db": -2.0,
        "min_pitch_semitones": -1.0,
        "min_snr_in_db": 8.0,
        "pitch_prob": 0.3,
        "rir_prob": 0.4
    },

    # ==========================================================
    # Feature Generation
    # ==========================================================
    "feature_generation_manifest": {
        "jast_a_name_like_pos_feature": {
            "input_audio_dirs": ["./data/positive"],
            "output_filename": "positive_features.npy",
            "use_background_noise": True,
            "use_rir": False,
            "augmentation_rounds": 10
        },
        "pos_val_feature": {
            "input_audio_dirs": ["./data/positive_val"],
            "output_filename": "positive_features_val.npy",
            "use_background_noise": True,
            "use_rir": False,
            "augmentation_rounds": 20
        },
        "negatives_1": {
            "input_audio_dirs": ["./data/negative"],
            "output_filename": "negative_features.npy",
            "use_background_noise": True,
            "use_rir": False,
            "augmentation_rounds": 10
        },
        "negatives_2": {
            "input_audio_dirs": ["./data/negative_phoneme"],
            "output_filename": "hard_negative_features.npy",
            "use_background_noise": True,
            "use_rir": False,
            "augmentation_rounds": 1
        }
    },

    "background_paths_duplication_rate": [1],

    # ==========================================================
    # Validation
    # ==========================================================
    "val_miss_weight": 4.0,
    "val_fp_weight": 1.0,
    "validation_batch_size": 256,
    "validation_smoothing_window": 3,
    "val_early_stopping_patience": 5000,

    # ==========================================================
    # Curriculum Learning
    # ==========================================================
    "hardness_ema_alpha": 0.05,
    "hardness_floor": 0.05,
    "hardness_reset_interval": 5000,
    "hardness_reset_decay": 0.5,
    "checkpoint_averaging_top_k": 5,

    # ==========================================================
    # Checkpointing & Debug
    # ==========================================================
    "checkpointing": {
        "enabled": True,
        "interval_steps": 1000,
        "limit": 2
    },

    "early_stopping_patience": 0,
    "min_delta": 0.0001,
    "ema_alpha": 0.01,

    "onnx_opset_version": 17,
    "show_training_summary": True,
    "debug_mode": True,

    # ==========================================================
    # Pipeline Control
    # ==========================================================
    "transform_clips": True, # ⚠️ Don't forget to turn it off after finished.
    "generate_clips": True,  # ⚠️ Don't forget to turn it off after finished.
    "train_model": True,
    "overwrite": False
}

config_path = "./config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)

print("⚠️[NOTE] Set `transform_clips: False` and re-run this cell unless feature (.npy) re-generation is required.\n")
print("config.yaml written successfully")

**Run Training!**

In [ ]:
# @title Step 3.2: Run the Magic Command! 🚀
# This command will do everything: augment data, extract features, and train the model.
# It might take some time depending on the hardware (especially on a CPU).

from nanowakeword.trainer import train

args_list = [
    '--config_path', f'{config_path}',
]

print("Starting NanoWakeWord training...")

try:
    train(args_list)
    print("\n\nCONGRATULATIONS! (✿◕‿◕✿)")
    print("Your custom wake word model has been successfully trained!")

except Exception as e:
    print(f"\nAn error occurred during training: {e}")

## What's Next?

You have successfully trained your own custom wake word model!

You can now download the `.onnx` or `.pt` file from the `trained_models` directory (check the file browser on the left) and use it in your own applications.

For more advanced topics, such as using your own datasets or fine-tuning the configuration, please check out our full documentation on **[GitHub](https://github.com/arcosoph/nanowakeword)**.

---
## Step 4: Save Your Model to Google Drive

The final step is to save your trained model and performance graph to a safe and accessible place. Instead of a slow direct download, we will save the files directly to your Google Drive. This process is almost instantaneous.

Run the cells below to:
1.  Connect your Google Drive account.
2.  Copy all the trained files into a new folder named `nanowakeword_models` in your Drive.

In [ ]:
# @title Step 4.1: Connect to Google Drive
# This will ask for your permission to access your Google Drive.

from google.colab import drive

try:
    drive.mount('/content/drive')
    print("\nGoogle Drive connected successfully!")
except Exception as e:
    print(f"An error occurred while connecting to Google Drive: {e}")

In [ ]:
# @title Step 4.2: Copy Final Model and Artifacts to Google Drive 📂

import os
import shutil

# --- Configuration ---
# Get model_name and output_dir from the config_dict defined earlier
model_name = config_dict.get("model_name", "my_model")
output_dir = config_dict.get("output_dir", "./trained_models")

# --- Source and Destination Paths ---
# The source project directory containing all generated files
source_project_dir = os.path.join(output_dir, model_name)

# The destination folder in your Google Drive
drive_destination_dir = f"drive/MyDrive/nanowakeword_models/{model_name}"

# --- Start Copy Process ---
print("Starting the process to copy trained files to Google Drive...")

# Check if the source directory exists
if not os.path.exists(source_project_dir):
    print(f"\nERROR: Source directory not found at '{source_project_dir}'")
    print("This indicates that the training process did not create the expected output folder.")
    print("Please ensure the training step completed successfully before running this cell.")
else:
    # If an old folder exists in Drive, remove it to ensure a clean copy
    if os.path.exists(drive_destination_dir):
        print(f"🔄 Found an existing folder in Drive. Removing it for a fresh copy: '{drive_destination_dir}'")
        shutil.rmtree(drive_destination_dir)

    # --- Copy the entire project folder ---
    # This is much simpler and more reliable than copying individual files.
    # It preserves the professional directory structure.
    try:
        shutil.copytree(source_project_dir, drive_destination_dir)

        print("\n" + "="*50)
        print("✅ SUCCESS! All files have been saved to your Google Drive.")
        print("="*50)
        print(f"\nYour complete project, including the model and performance graphs, can be found in:")
        print(f"➡️ '{drive_destination_dir}'")

        # Optional: List the contents of the new folder in Drive for verification
        print("\nContents of the saved folder:")
        for root, dirs, files in os.walk(drive_destination_dir):
            level = root.replace(drive_destination_dir, '').count(os.sep)
            indent = ' ' * 4 * (level)
            print(f"{indent}{os.path.basename(root)}/")
            sub_indent = ' ' * 4 * (level + 1)
            for f in files:
                print(f"{sub_indent}{f}")

    except Exception as e:
        print(f"\nERROR: An unexpected error occurred during the copy process.")
        print(f"Details: {e}")